# Train and evaluate segmentation model

This notebook runs `train.py` and then evaluates one checkpoint with `evaluate.py` on the validation split.

Check `SPLIT_FILE`, `GEN_ROOT`, and `OUT_DIR` before launching training.

## 1. Optional: mount Google Drive

Run this cell only in Colab if the project or data live in Drive.

In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


## 2. Set project directory

In [3]:
%cd /content/drive/MyDrive/diffusion-segmentation

/content/drive/MyDrive/diffusion-segmentation


In [4]:
from pathlib import Path
import os

PROJECT_DIR = Path.cwd()
os.chdir(PROJECT_DIR)

print('Working directory:', Path.cwd())
print('train.py exists:', Path('train.py').exists())
print('evaluate.py exists:', Path('evaluate.py').exists())

Working directory: /content/drive/MyDrive/diffusion-segmentation
train.py exists: True
evaluate.py exists: True


## 3. Install dependencies

In [5]:
%pip install -q monai nibabel scipy tqdm pandas wandb matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 100.6 MB/s eta 0:00:00


## 4. Configure paths and check GPU

In [6]:
import torch

SPLIT_FILE = Path('atlas_train_val.csv')

print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))

print('split file:', SPLIT_FILE.resolve(), SPLIT_FILE.exists())


torch: 2.10.0+cu128
cuda available: True
gpu: NVIDIA L4
split file: /content/drive/MyDrive/diffusion-segmentation/atlas_train_val.csv True


## 5. Run training

For a quick smoke test, reduce workers and use a tiny temporary split file.

In [ ]:
# import subprocess

# CACHE_WORKERS = 4
# LOADER_WORKERS = 4
# GEN_RATIO = 0.0
# GEN_SEED = 40
# SAVE_EVERY = 5
# DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'

# train_cmd = [
#     'python', 'train.py',
#     '--split-file', str(SPLIT_FILE),
#     '--out-dir', str(OUT_DIR),
#     '--gen-root', str(GEN_ROOT),
#     '--gen-ratio', str(GEN_RATIO),
#     '--gen-seed', str(GEN_SEED),
#     '--cache-workers', str(CACHE_WORKERS),
#     '--loader-workers', str(LOADER_WORKERS),
#     '--device', DEVICE,
#     '--save-every', str(SAVE_EVERY),
#     '--show-progress',
# ]

# print(' '.join(train_cmd))
# result = subprocess.run(train_cmd, text=True)
# if result.returncode != 0:
#     raise RuntimeError(f'train.py failed with return code {result.returncode}')

## 6. Run validation evaluation

In [7]:

CKPT_GLOB = 'outputs/gen_seed_40/atlas+uncond0/unet3d_best_*.pt'

# Resolve the latest checkpoint dir so the inspect cell below has EVAL_DIR / EVAL_METRICS.
best_checkpoints = sorted(
    Path('.').glob(CKPT_GLOB),
    key=lambda path: path.stat().st_mtime,
)
EVAL_DIR = best_checkpoints[-1].parent
EVAL_METRICS = EVAL_DIR / 'eval_metrics.json'


In [8]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="monai")

!python evaluate.py \
--checkpoint "{CKPT_GLOB}" \
--split-file "{SPLIT_FILE}" \
--path-prefix "/content/drive/MyDrive" \
--brain-mask-dir "atlas/strip_brain_mask" \
--vis-count 5


<frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
2026-05-03 23:30:51.263787: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-03 23:30:51.336283: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
[Split] loaded train=524, val=131 from /content/drive/MyDrive/diffusion-segmentation/atlas_train_val.csv
/usr/local/lib/python3.12/dist-packages/monai/utils/deprecate_utils.

## 7. Inspect outputs

In [ ]:
import json
import pandas as pd

print('Output directories:')
for path in sorted(OUT_DIR.parent.glob(OUT_DIR.name + '*')):
    print(' -', path)

summary_files = sorted(OUT_DIR.parent.glob(OUT_DIR.name + '*/run_summary.json'))
if summary_files:
    latest_summary = summary_files[-1]
    print('\\nLatest summary:', latest_summary)
    print(json.dumps(json.loads(latest_summary.read_text()), indent=2))

if EVAL_METRICS.exists():
    metrics = json.loads(EVAL_METRICS.read_text())
    print('\\nEvaluation metrics:', EVAL_METRICS)
    print('Dice:', metrics.get('dice'))
    print('IoU:', metrics.get('iou'))

registry = OUT_DIR.parent / 'experiment_registry.csv'
if registry.exists():
    display(pd.read_csv(registry).tail())